In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

DOC_FOLDER = Path(r"E:\Data\dat_dai_gpt")

uploaded_docs = []

for path in DOC_FOLDER.glob("*.txt"):
    file = client.files.create(
        file=open(path, "rb"),
        purpose="assistants"   # storage only
    )
    uploaded_docs.append({
        "file_id": file.id,
        "name": path.name
    })

print("Uploaded:", uploaded_docs)


In [4]:
def generate_prompt(question):


In [5]:
import pandas as pd

file_folder = r"E:\Data\dat_dai_gpt"
excel_path = r"E:\Github\LawAssistant\scripts\test_QA_notebookLM\facebook_question.xlsx"
output_csv = r"E:\Github\LawAssistant\scripts\test_QA_notebookLM\question_prompt.csv"

df = pd.read_excel(excel_path)

rows = []
for _, row in df.iterrows():
    question = row["Câu hỏi"]
    if pd.isna(question):
        continue

    system_prompt, user_prompt = generate_prompt(question)

In [2]:
import pandas as pd
import re

excel_path = r"E:\Github\LawAssistant\scripts\test_QA_notebookLM\facebook_question.xlsx"

def remove_markdown_and_dividers(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Remove fenced code blocks
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL)

    # Remove inline code
    text = re.sub(r"`([^`]*)`", r"\1", text)

    # Remove markdown headings
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)

    # Remove bold / italic
    text = re.sub(r"(\*\*|\*|__)(.*?)\1", r"\2", text)

    # Remove markdown links
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Remove divider lines (---, ***, ___)
    text = re.sub(r"^\s*([-*_]){3,}\s*$", "", text, flags=re.MULTILINE)

    # Remove bullet markers
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.MULTILINE)

    # Normalize whitespace
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = text.strip()

    return text

# Load Excel
df = pd.read_excel(excel_path)

# Clean column in place
df["Chat GPT"] = df["Chat GPT"].apply(remove_markdown_and_dividers)

# Save back to the same Excel file
df.to_excel(excel_path, index=False)